# Cross-Entropy

**Audience:** students who know entropy and are about to train classifiers.

**What you will build:** the cross-entropy loss from first principles: its link
to KL divergence, the binary and multiclass forms, a numerically stable softmax,
a small logistic-regression training loop, and a comparison with squared error.

## Learning objectives

1. define $H(p,q)$ and explain why minimising it matches minimising KL;
2. derive binary cross-entropy and the one-hot negative log-likelihood;
3. implement a stable softmax and multiclass cross-entropy;
4. train a logistic model by gradient descent on cross-entropy;
5. explain why cross-entropy beats squared error for confident wrong answers.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy as scipy_entropy

rng = np.random.default_rng(0)
plt.rcParams.update(
    {
        "figure.figsize": (8, 5),
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

print("Setup complete")

Setup complete


## 1. Definition and the KL connection

For a true distribution $p$ and a model $q$,

$$H(p,q) = -\sum_x p(x)\log q(x) = H(p) + D_{\mathrm{KL}}(p\,\|\,q).$$

The entropy $H(p)$ is constant with respect to the model, so minimising
cross-entropy over $q$ is equivalent to minimising KL. This is why cross-entropy
is the standard loss for probabilistic classifiers.

In [ ]:
def binary_cross_entropy(p, q, eps=1e-12):
    """H(p,q) for Bernoulli laws, in nats (vectorised over q)."""
    q = np.clip(np.asarray(q, dtype=float), eps, 1 - eps)
    return -p * np.log(q) - (1 - p) * np.log(1 - q)


p0 = 0.7
qs = np.linspace(0.01, 0.99, 400)
ce = binary_cross_entropy(p0, qs)
h = scipy_entropy([p0, 1 - p0])
kl = np.array([p0 * np.log(p0 / q) + (1 - p0) * np.log((1 - p0) / (1 - q)) for q in qs])

fig, ax = plt.subplots()
ax.plot(qs, ce, lw=2, label="H(p,q)")
ax.axvline(p0, color="tab:red", ls="--", label="q = p")
ax.set(xlabel="q", ylabel="nats", title="Binary cross-entropy is minimised at q = p")
ax.legend()
plt.show()

print(f"H(p) = {h:.6f} nats")
print(f"max |H(p,q) - H(p) - KL(p||q)| = {np.max(np.abs(ce - h - kl)):.2e}")

## 2. One-hot targets give negative log-likelihood

If the true label is class $y$, the target distribution is one-hot and
$H(p,q)=-\log q_y$. Minimising it over the model is maximum likelihood:
correct answers with high predicted probability get a small loss, while confident
mistakes are punished severely.

In [ ]:
q = np.linspace(1e-3, 1.0, 400)
nll = -np.log(q)
fig, ax = plt.subplots()
ax.plot(q, nll, lw=2)
ax.axvline(1.0, color="tab:red", ls="--", label="correct with certainty")
ax.set(xlabel="predicted probability of the true class", ylabel="-log q",
       title="Negative log-likelihood of a one-hot target")
ax.legend()
plt.show()
print(f"q = 1.0 -> loss {nll[-1]:.3f}; q = 0.01 -> loss {-np.log(0.01):.3f}")

## 3. Multiclass cross-entropy and a stable softmax

For logits $z_1,\ldots,z_k$ the model probabilities are
$q_i=\exp(z_i)/\sum_j\exp(z_j)$. Exponentiating large logits overflows, so we
subtract $\max_j z_j$ first: softmax is invariant to a constant shift. For a
batch with labels `y`, cross-entropy is `-log(q[np.arange(n), y])`.

In [ ]:
def softmax(logits):
    z = logits - np.max(logits, axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)


def categorical_cross_entropy(logits, y, eps=1e-12):
    probs = softmax(logits)
    return -np.log(probs[np.arange(len(y)), y] + eps)


logits = np.array([[2.0, 1.0, 0.1],
                   [0.1, 3.0, 0.2]])
y = np.array([0, 1])
probs = softmax(logits)
print("probabilities:\n", np.round(probs, 4))
print("cross-entropies:", np.round(categorical_cross_entropy(logits, y), 4))
print("shift invariance:", np.allclose(softmax(logits), softmax(logits + 100)))

## 4. Training a logistic model by gradient descent

For $q=\sigma(z)$ and $z=\mathbf{X}\mathbf{w}$, the average cross-entropy has the
remarkably simple gradient $\nabla_{\mathbf{w}}J=\mathbf{X}^\top(q-y)/n$. We use it
to fit a two-feature classifier and plot the loss curve.

In [ ]:
X = rng.normal(size=(200, 2))
y = (X[:, 0] + 0.5 * X[:, 1] + rng.normal(0, 0.3, 200) > 0).astype(int)
Xb = np.column_stack([np.ones(len(X)), X])

w = np.zeros(3)
eta, epochs = 0.5, 400
losses = []
for _ in range(epochs):
    z = Xb @ w
    p = 1 / (1 + np.exp(-np.clip(z, -30, 30)))
    grad = Xb.T @ (p - y) / len(y)
    w -= eta * grad
    losses.append(np.mean(-y * np.log(p + 1e-12) - (1 - y) * np.log(1 - p + 1e-12)))

accuracy = np.mean((Xb @ w > 0).astype(int) == y)
fig, ax = plt.subplots()
ax.plot(losses)
ax.set(xlabel="epoch", ylabel="mean cross-entropy", title="Gradient descent on cross-entropy")
plt.show()
print(f"loss {losses[0]:.4f} -> {losses[-1]:.4f}, accuracy {accuracy:.3f}")

## 5. Why not squared error?

With target $y=1$ and prediction $q=\sigma(z)$, cross-entropy is $-\log q$ while
squared error is $(1-q)^2$. For a very confident mistake ($z\ll0$) cross-entropy
grows without bound and its gradient stays informative, whereas squared error
saturates and the gradient vanishes.

In [ ]:
z = np.linspace(-6, 6, 400)
q = 1 / (1 + np.exp(-z))
ce = -np.log(q + 1e-12)
mse = (1 - q) ** 2

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(z, ce, lw=2, color="tab:blue")
axes[0].set(title="cross-entropy", xlabel="logit z", ylabel="loss")
axes[1].plot(z, mse, lw=2, color="tab:orange")
axes[1].set(title="squared error", xlabel="logit z", ylabel="loss")
fig.tight_layout()
plt.show()
print(f"at z = -6: CE = {ce[0]:.2f}, MSE = {mse[0]:.2f}")

## Exercises

1. Show that binary cross-entropy is minimised at $q=p$ by differentiating it.
2. Verify numerically that softmax probabilities sum to one for random logits.
3. Add $\ell_2$ regularisation to the training loop and observe the weights.
4. Compare convergence speed of cross-entropy and squared error on the same data.
5. Use `scipy.optimize.minimize` with an analytic gradient to fit the same model.